# LangChain RAG pipeline (modular)

Hadith RAG over `data/dorar_hadith_full_batch_2.csv`. Each section below is one pipeline stage; change **`CONFIG`** in the next cell to swap models, chunking, or retrieval without touching the rest.

**Run order:** Config → Imports → Load → Split → Embeddings → Vector store → Retriever → LLM → Prompt → Chain → Query.

**Requirements:** `pip install -r requirements.txt` (optional: copy `.env.example` to `.env` for API keys).

In [ ]:
!python -m pip install -r requirements.txt

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
from pathlib import Path

# --- Change settings here only ---
CONFIG = {
  "paths": {
    "data_csv": Path("data/dorar_hadith_full_batch_2.csv"),
    "chroma_dir": Path("chroma_db"),
    "collection_name": "hadith_rag",
  },
  "data": {
    "max_rows": 500,          # None = full CSV (large). Start small while experimenting.
    "text_columns": [         # Columns merged into each document body
      "hadith_1", "rawy_1", "mohadth_1", "source_1", "hokm_1",
      "categories", "sharh",
    ],
    "metadata_columns": ["page_id", "url", "categories"],
  },
  "chunking": {
    "chunk_size": 800,
    "chunk_overlap": 120,
    "separators": ["\n\n", "\n", ". ", " ", ""],
  },
  "embeddings": {
    "provider": "huggingface",  # "huggingface" | "openai"
    "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "openai_model": "text-embedding-3-small",
  },
  "vector_store": {
    "persist": True,
    "reset_on_build": False,  # True = delete collection and re-index
  },
  "retriever": {
    "search_type": "similarity",  # "similarity" | "mmr"
    "k": 4,
    "fetch_k": 12,                # used when search_type == "mmr"
    "lambda_mult": 0.5,
  },
  "llm": {
    "provider": "qrok",       # "openai" | "ollama"
    "openai_model": "gpt-4o-mini",
    "ollama_model": "llama3.2",
    "groq_model"  :"llama-3.3-70b-versatile",
    "temperature" : 0.1,
  },
  "prompt": {
    "language": "ar",           # answer language hint for the model
    "system_role": (
      "أنت مساعد متخصص في الحديث النبوي. أجب من السياق المسترجع فقط. "
      "إذا لم تجد جواباً في السياق، قل ذلك بوضوح ولا تخترع."
    ),
  },
}

PROJECT_ROOT = Path(".").resolve()
CONFIG["paths"]["data_csv"] = PROJECT_ROOT / CONFIG["paths"]["data_csv"]
CONFIG["paths"]["chroma_dir"] = PROJECT_ROOT / CONFIG["paths"]["chroma_dir"]

In [ ]:
import os
import shutil
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

def cfg(*keys: str) -> Any:
    """Read nested CONFIG values, e.g. cfg('retriever', 'k')."""
    node = CONFIG
    for key in keys:
        node = node[key]
    return node

In [ ]:
def row_to_page_content(row: pd.Series) -> str:
    parts = []
    for col in cfg("data", "text_columns"): # for each column
        if col in row.index:
            val = str(row[col]).strip()
            if val and val.lower() != "nan":
                parts.append(f"{col}: {val}")
    return "\n".join(parts)

def load_hadith_documents() -> list[Document]:
    csv_path = cfg("paths", "data_csv")
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    max_rows = cfg("data", "max_rows")
    if max_rows:
        df = df.head(max_rows)

    docs: list[Document] = []
    for _, row in df.iterrows():
        text = row_to_page_content(row)
        if not text.strip():
            continue
        metadata = {
            col: row[col] for col in cfg("data", "metadata_columns") if col in row.index and pd.notna(row[col])
        }
        docs.append(Document(page_content=text, metadata=metadata))

    print(f"Loaded {len(docs)} documents from {csv_path.name} ({len(df)} rows read)")
    return docs


raw_documents = load_hadith_documents()
raw_documents[0].page_content[:400] if raw_documents else "No documents"

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=cfg("chunking", "chunk_size"),
    chunk_overlap=cfg("chunking", "chunk_overlap"),
    separators=cfg("chunking", "separators"),
)

chunks = text_splitter.split_documents(raw_documents)
print(f"Split into {len(chunks)} chunks (avg ~{sum(len(c.page_content) for c in chunks) // max(len(chunks), 1)} chars)")
chunks[0].page_content[:300] if chunks else None

In [ ]:
def build_embeddings():
    provider = cfg("embeddings", "provider")
    if provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=cfg("embeddings", "openai_model"))
    if provider == "huggingface":
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name=cfg("embeddings", "model_name"))
    raise ValueError(f"Unknown embeddings provider: {provider}")


embeddings = build_embeddings()
# Quick sanity check (optional; comment out on slow machines)
# len(embeddings.embed_query("اختبار"))
print(f"Embeddings ready: {cfg('embeddings', 'provider')} / {cfg('embeddings', 'model_name') if cfg('embeddings', 'provider') == 'huggingface' else cfg('embeddings', 'openai_model')}")

In [ ]:
from langchain_chroma import Chroma

chroma_dir = cfg("paths", "chroma_dir")
collection = cfg("paths", "collection_name")

if cfg("vector_store", "reset_on_build") and chroma_dir.exists():
    shutil.rmtree(chroma_dir)
    print(f"Removed {chroma_dir}")

vectorstore = Chroma(
    collection_name=collection,
    embedding_function=embeddings,
    persist_directory=str(chroma_dir) if cfg("vector_store", "persist") else None,
)

def _collection_has_vectors(vs: Chroma) -> bool:
    data = vs.get(limit=1)
    return bool(data.get("ids"))


# Index only if collection is empty (re-run safe)
if not _collection_has_vectors(vectorstore):
    vectorstore.add_documents(chunks)
    print(f"Indexed {len(chunks)} chunks into '{collection}'")
else:
    n = len(vectorstore.get().get("ids", []))
    print(f"Using existing index: {n} vectors in '{collection}'")

vectorstore

In [ ]:
def build_retriever():
    search_type = cfg("retriever", "search_type")
    k = cfg("retriever", "k")

    if search_type == "similarity":
        return vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})
    if search_type == "mmr":
        return vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": k,
                "fetch_k": cfg("retriever", "fetch_k"),
                "lambda_mult": cfg("retriever", "lambda_mult"),
            },
        )
    raise ValueError(f"Unknown search_type: {search_type}")


retriever = build_retriever()
print(f"Retriever: {cfg('retriever', 'search_type')}, k={cfg('retriever', 'k')}")

In [ ]:
def build_llm():
    provider = cfg("llm", "provider")
    temperature = cfg("llm", "temperature")

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=cfg("llm", "openai_model"), temperature=temperature)
    if provider == "ollama":
        from langchain_community.chat_models import ChatOllama
        return ChatOllama(model=cfg("llm", "ollama_model"), temperature=temperature)
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=cfg("llm", "groq_model"), temperature=temperature)
        # max_tokens=1024
    raise ValueError(f"Unknown llm provider: {provider}")


llm = build_llm()
print(f"LLM: {cfg('llm', 'provider')}")

In [ ]:
def format_docs(docs: list[Document]) -> str:
    blocks = []
    for i, doc in enumerate(docs, 1):
        meta = ", ".join(f"{k}={v}" for k, v in doc.metadata.items())
        blocks.append(f"[{i}] ({meta})\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)


RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", cfg("prompt", "system_role")),
    (
        "human",
        "السياق المسترجع:\n{context}\n\n"
        "السؤال: {question}\n\n"
        f"أجب باللغة: {cfg('prompt', 'language')}. اذكر page_id عند الاقتباس إن وُجد.",
    ),
])

prompt = RAG_PROMPT
prompt

In [ ]:
# LCEL chain: question -> retrieve -> format -> prompt -> llm -> text
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Optional: inspect retrieval only (no LLM cost)
def retrieve(question: str, k: int | None = None):
    docs = retriever.invoke(question)
    if k:
        docs = docs[:k]
    return docs

rag_chain

In [ ]:
def ask(question: str, *, show_sources: bool = True) -> str:
    if show_sources:
        print("--- Retrieved chunks ---")
        for doc in retrieve(question):
            print(f"page_id={doc.metadata.get('page_id')} | {doc.page_content[:120]}...")
        print("--- Answer ---")
    return rag_chain.invoke(question)


# Example (set OPENAI_API_KEY in .env if using OpenAI)
QUESTION = "ما حكم سجود السهو إذا زاد الإمام في الصلاة؟"
# answer = ask(QUESTION)
# print(answer)

## Customization cheat sheet

| Goal | Change in `CONFIG` |
|------|-------------------|
| Use full dataset | `"max_rows": None` |
| Rebuild vector DB | `"reset_on_build": True` (run vector-store cell once) |
| More context per answer | Increase `retriever.k` or `chunk_size` |
| Diverse retrieval | `"search_type": "mmr"` |
| Local LLM | `"llm": {"provider": "ollama", ...}` + run Ollama |
| OpenAI embeddings | `"embeddings": {"provider": "openai", ...}` |
| Different fields in chunks | Edit `data.text_columns` / `metadata_columns` |
| Swap only the prompt | Edit `prompt.system_role` or the `RAG_PROMPT` cell |

**Swap a component:** re-run from that cell downward (e.g. new embeddings → vector store → retriever → chain).